# FastAPI from Frictionless Data Packages Devlopment Log

## 2026-02-26 - @jinskeep-morpc

### 1. Create the skeleton for the package using morpc-py and development already started in morpc-purpleair-model as the basis. 

- Had some issues with getting myst pages to work with the github action. 
- The issues was that the ipynbs for demos and devlog were empty, which created an issue when making the pages.
- Added titles to both and it worked.
- See [https://jinskeep-morpc.github.io/fastapi-from-frictionless/](https://jinskeep-morpc.github.io/fastapi-from-frictionless/)


### 2. Creating SQLmodel from frictionless

- I am going to use the schemas from the [morpc-purpleair-model](https://github.com/morpc/morpc-purpleair-model) as the first use case as this is already mostly developed.

> [!NOTE]
> I will be working primarily in the doc folder then moving things over to the package folder when I refactor.

1. I want to create some mock resource files and then combine then into a data package.
    - Create a yaml file for resources. 
    - Work around for creating a resource yaml without actually creating the file. 
        - Try leaving path blank for now and not validate on the creation of the resource?
        - Nevermind, I just need to create the schemas and create the resource file when I have models and database created. 
2. Creating schemas


In [ ]:
import frictionless

deployment_schema = frictionless.Schema('data/deployment.schema.yaml')

In [ ]:
deployment_schema

#### Creating type map for frictionless to pydantic data types

> [!NOTE]
> [Frictionless types](https://datapackage.org/standard/table-schema/#field-types)

> [!NOTE]
> [Pydantic types](https://docs.pydantic.dev/1.10/usage/types/#standard-library-types)

In [ ]:
from frictionless import Schema, extract, fields

extract([["name"], [9]], schema=Schema(fields=[fields.StringField(name='name', format="default")]))

In [ ]:
extract([["name"], ["http://morpc.org"]], schema=Schema(fields=[fields.StringField(name='name', format="uri")]))

In [ ]:
extract([["name"], ["not_a_uri"]], schema=Schema(fields=[fields.StringField(name='name', format="uri")]))

In [ ]:
extract([["name"], ["dataandmaps@morpc.org"]], schema=Schema(fields=[fields.StringField(name='name', format="email")]))

In [ ]:
from uuid import uuid4

extract([["name"], [f"{uuid4()}"]], schema=Schema(fields=[fields.StringField(name='name', format="uuid")]))

In [ ]:
import base64

bb = base64.b64encode(bytes("This is a test of binary strings", 'utf-8'))
extract([["name"], [f"{bb}"]], schema=Schema(fields=[fields.StringField(name='name', format="binary")]))

In [ ]:
extract([["name"], [10.2]], schema=Schema(fields=[fields.NumberField(name='name')]))

In [ ]:
frictionless.settings.DEFAULT_FIELD_CANDIDATES

In [ ]:
type_map = {
    "string": {
        "default": "str",
        "email": "EmailStr",
        "uri": "AnyUrl",
        "binary": "bytes",
        "uuid": "UUID"
    },
    "number": {
        "default": "float"
    },
    "integer": {
        "default": "int"
    },
    "boolean": {
        "default": "bool"
    },
    "object": {
        "default": "Json[Any]"
    },
    "array": {
        "default": "List[Any]"
    },
    "datetime": {
        "default": "datetime"
    },
    "date": {
        "default": "date"
    },
    "time": {
        "default": "time"
    },
    "year": {
        "default": "int"
    },
    "duration": {
        "default": "timedelta"
    },
    "geopoint": {
        "default": "Geometry('POINT')"
    },
    "geojson": {
        "default": "Geometry('GEOMETRY')"
    }
}

In [ ]:
header = """
from typing import Optional, List
from uuid import UUID
from sqlalchemy import DateTime
from sqlmodel import Field, Relationship, SQLModel
from datetime import date, datetime, timezone, time, timedelta
from pydantic import EmailStr, AnyUrl, Json
from geoalchemy2.types import Geometry

def utcnow():
    '''Returns the current time in UTC.'''
    return datetime.now(timezone.utc)

class TimestampMixin: # https://www.davidmuraya.com/blog/reusable-sqlmodel-mixins/
    '''A mixin to add created_at and updated_at timestamp fields to a model.'''

    created_at: datetime = Field(
        default_factory=utcnow,
        nullable=False,
        sa_type=DateTime(timezone=True)
    )
    updated_at: datetime = Field(
        default_factory=utcnow,
        nullable=False,
        sa_column_kwargs={"onupdate": utcnow},
        sa_type=DateTime(timezone=True)
    )
"""

In [ ]:
import os
import frictionless
folder = './data'
schema_paths = [x for x in os.listdir(folder) if x.endswith('schema.yaml')]


In [ ]:
models = []
for filename in schema_paths:
    filepath = os.path.join(folder, filename)
    name = filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
    schema = frictionless.Schema(filepath)
    foreign_keys = [x['fields'][0] for x in schema.foreign_keys]

    basemodel_fields = []
    auto_id = "id" in schema.primary_key
    for field in schema.field_names:
        field = schema.get_field(field)

        if (field.name == 'id') & (auto_id == True):
            continue
        else:
            field_string = ""
            field_string += f"{field.name}: "
            field_string += f"{type_map[field.type][field.format]}"  

            if not 'required' in field.constraints:
                field_string += " | None"
                required = False

            if field.name in schema.primary_key:
                field_string += " = Field(primary_key = True)"
            
            if field.name in foreign_keys:
                field_string += f" = Field({"default=None, " if required else ""}foreign_key='{field.name.replace('_', '.')}')"

            basemodel_fields.append(field_string)

    relationships = []
    for other_filename in schema_paths:
        if other_filename != filename:
            other_filepath = os.path.join(folder, other_filename)
            other_name = other_filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
            other_schema = frictionless.Schema(other_filepath)
            if len(other_schema.foreign_keys) > 0:
                for fk in other_schema.foreign_keys:
                    if fk['reference']['resource'] == name.lower():
                        relationships.append(other_name)

    basemodel_string = f"""class {name}Base(SQLModel):
    {"\n    ".join(basemodel_fields)}
"""

    tablemodel_string = f"""class {name}({name}Base, TimestampMixin, table=True):
"""
    if auto_id == True:
        tablemodel_string += "    id: int | None = Field(default=None, primary_key=True)\n"
    if len(relationships) > 0:
        for relationship in relationships:
            tablemodel_string += f"    {relationship.lower()}s: list['{relationship}'] | None = Relationship(back_populates='{name.lower()}s')\n"
    if len(foreign_keys) > 0:
        for fk in foreign_keys:
            tablemodel_string += f"    {fk.split('_')[0]}s: list['{fk.split('_')[0].capitalize()}'] | None = Relationship(back_populates='{name.lower()}s')\n" 
    if (auto_id == False) & (len(relationships) == 0) & (len(foreign_keys) == 0):
        tablemodel_string += '    pass\n'


    createmodel_string = f"""class {name}Create({name}Base):
    pass
"""
    
    updatemodel_string = f"""class {name}Update({name}Base):\n"""
    for field in basemodel_fields:
        if not 'primary_key' in field:
            if " = " in field:
                field = field.split(" = ")[0]
            if not ' | None' in field:
                updatemodel_string += f"    {field} | None\n"
            else:
                updatemodel_string +=  f"    {field}\n"

    publicmodel_string = f"""class {name}Public({name}Base):{"\n    id: int" if auto_id == True else ''}
    created_at: datetime
    updated_at: datetime
"""
    
    relationshipsmodel_string = f""

    if len(foreign_keys) > 0:
        relationshipsmodel_string += f"""class {name}PublicWithAll({name}Public):\n"""
        for fk in foreign_keys:
            relationshipsmodel_string += f"    {fk.split("_")[0]}s: list['{fk.split('_')[0].capitalize()}'] | None\n"


    model_file = f"""
## {name} models
{basemodel_string}
{tablemodel_string}
{createmodel_string}
{publicmodel_string}
{relationshipsmodel_string}
{updatemodel_string}
    """
    models.append(model_file)



In [ ]:
with open('models.py', 'w') as file:
    file.write("".join([header] + models))

In [ ]:
filename = 'test.db'
database_string = f"""
from sqlmodel import SQLModel, create_engine

sqlite_filename = '{filename}'
sqlite_url = f"sqlite:///{{sqlite_filename}}"

connect_args = {{'check_same_thread': False}}
engine = create_engine(sqlite_url, echo=True, connect_args=connect_args)

def create_db_and_tables():
    SQLModel.metadata.create_all(engine)
"""
print(database_string)
with open('database.py', 'w') as file:
    file.write(database_string)

In [ ]:
app_header = """
# app.py
from fastapi import Depends, FastAPI, HTTPException, Query
import fastapi
from fastapi_querybuilder import QueryBuilder
from sqlalchemy import text
from sqlmodel import Session, select
from .database import create_db_and_tables, engine
from .models import *
from sqlalchemy.ext.asyncio import AsyncSession

# Initiate app
app = FastAPI()

# Dependencies
@app.on_event('startup')
def on_startup():
    create_db_and_tables()

def get_session():
    with Session(engine) as session:
        yield session
        
# @app.get('/')
# def read_schema(*, session: Session = Depends(get_session)):
#     return {name.lower()}s
"""

In [ ]:
import os
folder = './data'
schema_paths = [x for x in os.listdir(folder) if x.endswith('schema.yaml')]
endpoints = []

for filename in schema_paths:
    filepath = os.path.join(folder, filename)
    name = filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
    schema = frictionless.Schema(filepath)
    foreign_keys = [x['fields'][0] for x in schema.foreign_keys]
    
    post_string = f"""
# {name} requests
@app.post('/{name.lower()}s/', response_model={name}Public)
def create_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}: {name}Create):
    {name.lower()} = {name}.model_validate({name.lower()})
    session.add({name.lower()})
    session.commit()
    session.refresh({name.lower()})
    return {name.lower()}"""
    
    relationships = []
    for other_filename in schema_paths:
        if other_filename != filename:
            other_filepath = os.path.join(folder, other_filename)
            other_name = other_filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
            other_schema = frictionless.Schema(other_filepath)
            if len(other_schema.foreign_keys) > 0:
                for fk in other_schema.foreign_keys:
                    if fk['reference']['resource'] == name.lower():
                        relationships.append(other_name)

    pk = schema.primary_key[0]

    getall_string = f"""
@app.get('/{name.lower()}s/', response_model=list[{f'{name}PublicWithAll' if len(foreign_keys)>0 else f'{name}Public'}])
def read_{name.lower()}s(*, session: Session = Depends(get_session)):
    {name.lower()}s = session.exec(select({name})).all()
    return {name.lower()}s"""


    if len(foreign_keys) > 0:
        query_string = f"""
@app.get('/{name.lower()}s/query', response_model=list[{name}PublicWithAll])
async def query_{name.lower()}s(*, session: AsyncSession = Depends(get_session), query=QueryBuilder({name})):
    {name.lower()}s = session.execute(query)
    return {name.lower()}s.scalars().all()"""
    else:
        query_string = ""
    
    get_string = f"""
@app.get('/{name.lower()}s/{{{name.lower()}_{pk}}}', response_model={f'{name}PublicWithAll' if len(foreign_keys)>0 else f'{name}Public'})
def read_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}_{pk}: str):
    {name.lower()} = session.get({name}, {name.lower()}_{pk})
    if not {name.lower()}:
        raise HTTPException(status_code=404, detail='{name} not found.')
    return {name.lower()}"""
    
    update_string = f"""
@app.patch('/{name.lower()}s/{{{name.lower()}_{pk}}}', response_model={name}Public)
def update_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}_{pk}: str, {name.lower()}: {name}Update):
    db_{name.lower()} = session.get({name}, {name.lower()}_{pk})
    if not db_{name.lower()}:
        raise HTTPException(status_code=404, detail=f'{name} {{{name.lower()}_{pk}}} not found.')
    {name.lower()}_data = {name.lower()}.model_dump(exclude_unset=True)
    db_{name.lower()}.sqlmodel_update({name.lower()}_data)
    session.add(db_{name.lower()})
    session.commit()
    session.refresh(db_{name.lower()})
    return db_{name.lower()}"""
    
    delete_string = f"""
@app.delete('/{name.lower()}s/{{{name.lower()}_{pk}}}')
def delete_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}_{pk}: str):
    {name.lower()} = session.get({name}, {name.lower()}_{pk})
    if not {name.lower()}:
        raise HTTPException(status_code=404, detail=f'{name} {{{name.lower()}_{pk}}} not found.')
    session.delete({name.lower()})
    session.commit()
    return {{'ok': True}}"""

    endpoint_file = f"""
    {post_string}
    {getall_string}
    {get_string}
    {query_string}
    {update_string}
    {delete_string}
    """

    endpoints.append(endpoint_file)


In [ ]:
with open('app.py', 'w') as file:
    file.write("".join([app_header] + endpoints))

## Add excel workbook interface

load cleaned data from morpc-purpleair-model

In [ ]:
import pandas as pd

sheets = {}
for sheet in pd.ExcelFile('data/sensor_tracking.xlsx').sheet_names:
    sheets[sheet] = (pd.read_excel('data/sensor_tracking.xlsx', sheet_name=sheet))

In [ ]:
sheets['sensors']

In [ ]:
import requests
from models import SensorCreate

In [ ]:
{**sheets['sensors'].loc[0]}

### Create initial empty excel workbook

In [ ]:
import frictionless
import pandas as pd
import os
import datetime

folder = 'data'
schemas = [file for file in os.listdir(folder) if file.endswith('schema.yaml')]

with pd.ExcelWriter(f'sensor_tracking {datetime.date.today()}.xlsx') as writer:
    for schema in schemas:
        name = schema.replace('.schema.yaml', '')
        schema = frictionless.Schema(os.path.join(folder, schema))
        fields = schema.field_names
        df = pd.DataFrame(data=[], columns=fields)
        df.set_index(schema.primary_key).to_excel(writer, sheet_name=name)
    for sheet in writer.sheets:
        writer.sheets[sheet].autofit()


In [ ]:

def get_model(name: str, type: str):
    import models

    all_models = dir(models)
    for model in all_models:
        if model.casefold() == name.rstrip('s').casefold():
            return getattr(models, f"{model}{type.capitalize()}")

In [ ]:
get_model('sensornotes', type='create')

In [ ]:
import os
import pandas as pd

def requests_post(server_url: str | os.PathLike, endpoint: str, model) -> pd.DataFrame:
    import os
    import requests
    import pandas as pd

    r = requests.post(f'{os.path.join(server_url, endpoint)}', data=model.model_dump())
    r.raise_for_status()
    json = r.json()
    r.close()

    return json

In [ ]:
sensor_create = get_create_model('sensor')
sensor_create = sensor_create(**sheets['sensors'].loc[1])

In [ ]:
requests_post(server_url='http://127.0.0.1:8000', endpoint='sensors', model=sensor_create)

In [ ]:
def requests_get_all(server_url: str | os.PathLike, endpoint: str) -> pd.DataFrame:
    import os
    import requests
    import pandas as pd

    r = requests.get(f'{os.path.join(server_url, endpoint)}')
    r.raise_for_status()
    json = r.json()
    r.close()

    return pd.DataFrame.from_dict(json)

In [ ]:
df = requests_get_all(server_url='http://127.0.0.1:8000', endpoint='sensors')

In [ ]:
df_index = df[df.columns[0]].to_list()

In [ ]:
row = {**sheets['sensors'].loc[1]}


In [ ]:
pk = [x for x in row.values()][0]

In [ ]:
[x for x in row.values()][0] in df_index

In [ ]:
row

In [ ]:
df.loc[df[df.columns[0]]==pk]

In [ ]:
current = df.loc[df[df.columns[0]]==pk].loc[1].to_dict()

In [ ]:
current

In [ ]:
for k,v in row.items():
    if current[k] == v:
        print('No Change')


In [ ]:
sensor_create.model_dump()

In [ ]:
import os
import pandas as pd

def requests_update(server_url: str | os.PathLike, endpoint: str, model) -> pd.DataFrame:
    import os
    import requests
    import pandas as pd

    pk = [x for x in model.model_dump().values()][0]
    r = requests.patch(f'{os.path.join(server_url, endpoint, pk)}', data=model.model_dump())
    r.raise_for_status()
    json = r.json()
    r.close()

    return json

In [ ]:
requests_update(server_url='http://127.0.0.1:8000', endpoint='sensors', model=sensor_create)

In [ ]:
server_url = 'http://127.0.0.1:8000/'

current_all = requests_get_all(server_url=server_url, endpoint=sheet)
current_all


In [ ]:
current_row = current_all.loc[current_all[current_all.columns[0]]==pk]
current_row

In [ ]:
current_row.loc[[x for x in current_row.index][0]]

In [ ]:
current_all.columns

In [ ]:
import fastapi

folder = 'data'
schemas = [file for file in os.listdir(folder) if file.endswith('schema.yaml')]

tracking_filename = 'sensor_tracking.xlsx'
server_url = 'http://127.0.0.1:8000/'
sheets = pd.ExcelFile(tracking_filename).sheet_names

for sheet in sheets:
    try:
        current_all = requests_get_all(server_url=server_url, endpoint=sheet)
    except Exception as e:
        print(f"{sheet} not in tables at {server_url}")
        raise fastapi.exceptions.RequestValidationError(e)

    if len(current_all) == 0:
        current_index = []
    else: 
        current_index = current_all[current_all.columns[0]].to_list()

    sensor_tracking = pd.read_excel(tracking_filename, sheet_name=sheet)
    
    for i, row in sensor_tracking.iterrows():
        row = {**row}
        row_pk = [x for x in row.values()][0]
        modelcreate = get_model(sheet, 'create')
        model = modelcreate(**row)
        if row_pk not in current_index:
            response = requests_post(server_url=server_url, endpoint=sheet, model=model)
        if row_pk in current_index:
            current_row = current_all.loc[current_all[current_all.columns[0]]==row_pk]
            current_row = current_row.loc[[x for x in current_row.index][0]].to_dict()
            for k,v in row.items():
                changed = False
                if current_row[k] == v:
                    continue
                else:
                    changed = True
            if changed:
                modelupdate = get_model(sheet, 'update')
                model = modelupdate(**row)
                response = requests_update(server_url=server_url, endpoint=sheet, model=model)


In [ ]:
updatemodel = get_model('sensornote', 'update')
updatemodel(**row).model_dump()

requests_update(server_url='http://127.0.0.1:8000', endpoint='sensornotes', pk=1, model=updatemodel(**row))

In [ ]:
sensor_tracking = pd.read_excel(tracking_filename, sheet_name='sensornotes')


In [ ]:
sensor_tracking

## Restart and rebuild using package

In [1]:
from morpc.logs import config_logs

config_logs('sensor_tracking.log', 'debug')

2026-03-06 16:00:37,627 | INFO | morpc.logs.config_logs: Set up logging save to file "sensor_tracking.log", log level "debug"


In [2]:
import fastapifromfrictionless

In [18]:
fastapifromfrictionless.models(folder='data').build().save('models.py')

2026-03-06 16:30:56,077 | INFO | fastapifromfrictionless.model.__init__: Building models for schemas from data: location.schema.yaml .link-deployment-contact.schema.yaml .hotspot.schema.yaml .sensor.schema.yaml .registration.schema.yaml .sensor-note.schema.yaml .contact.schema.yaml .deployment.schema.yaml
2026-03-06 16:30:56,079 | INFO | fastapifromfrictionless.model.models.data.build: Building model for location.schema.yaml
2026-03-06 16:30:56,087 | INFO | fastapifromfrictionless.model.models.data.build_model: Schema Location foreign keys ['deployment_name']
2026-03-06 16:30:56,117 | INFO | fastapifromfrictionless.model.models.data.build_model: Location is referenced by []
2026-03-06 16:30:56,119 | INFO | fastapifromfrictionless.model.models.data.build_model: {'name': 'address', 'type': 'string', 'constraints': {'required': True}} converted to address: str = Field(primary_key = True)
2026-03-06 16:30:56,120 | INFO | fastapifromfrictionless.model.models.data.build_model: {'name': 'zipc

2026-03-06 16:30:56,326 | INFO | fastapifromfrictionless.model.models.data.build_model: Registration is referenced by []
2026-03-06 16:30:56,328 | INFO | fastapifromfrictionless.model.build_model: Primary key is 'id'. Will add to table model with autoincrement.
2026-03-06 16:30:56,329 | INFO | fastapifromfrictionless.model.models.data.build_model: {'name': 'macaddr', 'type': 'string'} converted to macaddr: str | None
2026-03-06 16:30:56,330 | INFO | fastapifromfrictionless.model.models.data.build_model: {'name': 'reg_email', 'type': 'string', 'format': 'email'} converted to reg_email: EmailStr | None
2026-03-06 16:30:56,331 | INFO | fastapifromfrictionless.model.models.data.build_model: {'name': 'outside', 'type': 'boolean'} converted to outside: bool | None
2026-03-06 16:30:56,332 | INFO | fastapifromfrictionless.model.models.data.build_model: {'name': 'sensor_name', 'type': 'string'} converted to sensor_name: str | None = Field(foreign_key='sensor.name')
2026-03-06 16:30:56,333 | INF

In [4]:
fastapifromfrictionless.database('test.db').save('database.py')

2026-03-06 16:00:45,436 | INFO | fastapifromfrictionless.database.database.test.db.__init__: Building database from schema test.db
2026-03-06 16:00:45,438 | INFO | fastapifromfrictionless.database.database.test.db.save: Saving database file to database.py


In [14]:
fastapifromfrictionless.app('data').build().save('app.py')

In [ ]:
# import fastapifromfrictionless

# api = fastapifromfrictionless.API('dev', 'app.py')

In [ ]:

# import psutil

# def kill(proc_pid):
#     process = psutil.Process(proc_pid)
#     for proc in process.children(recursive=True):
#         proc.kill()
#     process.kill()

# kill(proc_pid=api.process.pid)


In [ ]:
# import os
# import signal

# os.killpg(os.getpgid(api.process.pid), signal.SIGTERM)

Ended up not creating a python call for the fastapi launch due to issues with terminating the process after it starts.

In [ ]:
import fastapifromfrictionless

fastapifromfrictionless.empty_excel('data', 'sensor_tracking.xlsx')

In [ ]:
# fastapifromfrictionless.update_from_excel(api_url='http://127.0.0.1:8000', excel_file='sensor_tracking_legacy.xlsx')

In [ ]:
import frictionless
import frictionless.formats
import os
from datetime import datetime


folder = 'data'
filename = 'sensor_tracking.xlsx'

cwd = os.getcwd()

os.chdir(folder)
schemas = [x for x in os.listdir(os.getcwd()) if x.endswith('schema.yaml')]
resources=[]

for schema in schemas:
    schema_name = schema.replace('.schema.yaml', "").replace('-','')
    resource = frictionless.Resource(name=schema_name, 
                                     path=filename,
                                     control=frictionless.formats.ExcelControl(sheet=schema_name),
                                     schema=f'{schema}')
    resource.infer(stats=True)
    resources.append(resource)

package = frictionless.Package(
    name=filename.split('.')[0],
    resources=resources,
    version='0.0.1',
    created=datetime.now().isoformat()
)

package.to_yaml(filename.split('.')[0]+'.package.yaml')

os.chdir(cwd)

In [ ]:
package = frictionless.Package('data/sensor_tracking.package.yaml')

In [ ]:
pd.DataFrame.from_records(package.resources[0].read_rows())

In [ ]:
import pandas as pd
with package.get_resource('registration') as resource:
    resource.index('sqlite:///data/test.db', name='registration')

In [ ]:
resource = frictionless.Package('data/sensor_tracking.package.yaml').get_resource('registration')
resource

In [ ]:
pd.DataFrame.from_records(resource.read_rows()).dtypes

In [ ]:

with frictionless.Resource(resource).open() as data:
    sensor_tracking = pd.DataFrame.from_records(data.read_rows())

In [ ]:
data = frictionless.Resource('sqlite:///data/test.db', control=frictionless.formats.SqlControl(table='registration')).extract()

In [ ]:
import frictionless
import logging
package = frictionless.Package('data/sensor_tracking.package.yaml')
resource = package.get_resource('deployment')

In [ ]:

with package.get_resource('deployment') as resource:
    for row in resource.cell_stream:
        if row[0] == None:
            break
        print(row)

In [ ]:

import requests 

r = requests.get('http://127.0.0.1:8000/sensornote/all')
r.content

In [ ]:
from fastapifromfrictionless.load import update_api_from_package

In [ ]:
update_api_from_package('http://127.0.0.1:8000', 'data/sensor_tracking.package.yaml')